In [4]:
# =============================================================
# 0.  Imports & paths
# =============================================================
import os, json, datetime, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
np.random.seed(42)

from scipy.stats import ks_2samp
from statsmodels.tsa.api import VAR
from sklearn.metrics import mean_squared_error

# ---- ! install timegan (first run only) ----------------------
# %pip install ydata-timeseries==1.2.2  # lightweight TimeGAN impl
# --------------------------------------------------------------

from ydata_synthetic.synthesizers.timeseries import TimeGAN
from ydata_synthetic.preprocessing.timeseries.utils import \
        real_data_loading, train_test_split_sequence

DATA_PATH = "./final_processed_data.csv"
OUT_DIR   = "./results/baseline/"
os.makedirs(OUT_DIR, exist_ok=True)

# =============================================================
# 1.  Load & train/val/test split
# =============================================================
df = pd.read_csv(DATA_PATH, index_col="Date", parse_dates=True)
features = df.columns.tolist()

# chronological split 70 / 15 / 15
n = len(df)
train_end   = int(0.7*n)
val_end     = int(0.85*n)

train = df.iloc[:train_end]
val   = df.iloc[train_end:val_end]
test  = df.iloc[val_end:]

print(f"train {train.shape}, val {val.shape}, test {test.shape}")

# =============================================================
# 2.  VAR(1)
# =============================================================
var = VAR(train)
var_res = var.fit(maxlags=1)
print(var_res.summary())

# 1‑step‑ahead forecast rolling through val+test
def rolling_forecast(model, history, steps):
    preds = []
    for i in range(steps):
        pred = model.forecast(y=history.values[-model.k_ar:], steps=1)
        preds.append(pred[0])
        history = pd.concat(
            [history, pd.DataFrame(pred, columns=model.names,
                                   index=[history.index[-1] + pd.DateOffset(months=1)])])
    return np.vstack(preds)

val_pred  = rolling_forecast(var_res, train.copy(), len(val))
test_pred = rolling_forecast(var_res, pd.concat([train, val]), len(test))

val_mse  = mean_squared_error(val,  val_pred)
test_mse = mean_squared_error(test, test_pred)
print(f"VAR  MSE  val={val_mse:.3f}  test={test_mse:.3f}")

# plot one variable
plt.figure(figsize=(7,3))
plt.plot(test.index, test['SP500_Returns'], label="real")
plt.plot(test.index, test_pred[:, features.index('SP500_Returns')],
         label="VAR forecast")
plt.legend(); plt.title("VAR(1) 1‑step forecast — S&P 500 returns")
plt.tight_layout(); plt.savefig(OUT_DIR+"var_sp500_forecast.png", dpi=200)

# =============================================================
# 3.  TimeGAN baseline  (ydata‑timeseries)
# =============================================================
SEQ_LEN = 12      # one‑year windows
BATCH   = 64
EPOCHS  = 300

# prepare 3‑D array [samples, seq_len, features]
train_seq, val_seq, test_seq = train_test_split_sequence(df.values,
                                                         train_end, val_end,
                                                         SEQ_LEN)

synth = TimeGAN(window_size=SEQ_LEN, batch_size=BATCH, n_epochs=EPOCHS,
                hidden_dim=24, gamma=1.5, noise_dim=32,
                learning_rate=5e-4, num_layers=2)

synth.train(train_seq)

# generate synthetic test‑sized set
synth_data = synth.generate(len(test_seq))

# flatten back to 2‑D (keep last step of each sequence)
synth_last = synth_data[:, -1, :]
test_last  = test_seq[:, -1, :]

# ---- Metrics -------------------------------------------------
def rmse(a,b): return np.sqrt(np.mean((a-b)**2))

timegan_rmse = rmse(test_last, synth_last)
ks = {features[i]: ks_2samp(test_last[:,i], synth_last[:,i]).statistic
      for i in range(len(features))}

print(f"TimeGAN RMSE on last‑step vectors: {timegan_rmse:.3f}")
print("KS distances real vs synth:")
print(ks)

# plot distribution of S&P returns real vs synth
plt.figure(figsize=(4,3))
sns.kdeplot(test_last[:,0], label="real")
sns.kdeplot(synth_last[:,0], label="TimeGAN")
plt.title("S&P 500 monthly return distribution")
plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR+"timegan_sp500_dist.png", dpi=200)

# =============================================================
# 4.  Save metrics
# =============================================================
metrics = dict(
    timestamp = datetime.datetime.now().isoformat(),
    var      = dict(MSE_val=val_mse, MSE_test=test_mse),
    timegan  = dict(RMSE=timegan_rmse, KS=ks)
)
json.dump(metrics, open(OUT_DIR+"baseline_metrics.json","w"), indent=2)
print("saved metrics to", OUT_DIR+"baseline_metrics.json")


ModuleNotFoundError: No module named 'ydata_synthetic'